# Lab 2 — Decoding and LoRA Fine-Tuning

Load a real instruct model, compare every decoding strategy, inspect how temperature reshapes the logit distribution, apply chat templating correctly, then run a full LoRA fine-tune loop and verify generation before and after.

Runs on Google Colab — set Runtime → Change runtime type → GPU before running (this lab needs a GPU for the fine-tuning step; CPU/MPS also work but are much slower).

The full write-up and stack alternatives live in the lesson: `11_lab_llms.md`.

In [ ]:
!pip install -q torch transformers peft trl datasets accelerate

## Load the model

**Model:** `Qwen/Qwen2.5-0.5B-Instruct` — ~1 GB fp16. Runs on CPU (slow), MPS (M-series Mac), or CUDA.

In [ ]:
import random
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM

random.seed(42); np.random.seed(42); torch.manual_seed(42)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(42)

device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

MODEL_ID  = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16 if device in ("cuda", "mps") else torch.float32,
    device_map=device,
)
model.eval()
print(f"device={device}  params={sum(p.numel() for p in model.parameters())/1e6:.0f}M")

## Part A — Decoding Strategies

Compare greedy, temperature, top-k, and top-p (nucleus) sampling on the same prompt.

In [ ]:
def gen(prompt: str, **kw) -> str:
    ids = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model.generate(**ids, pad_token_id=tokenizer.eos_token_id, **kw)
    return tokenizer.decode(out[0, ids["input_ids"].shape[1]:], skip_special_tokens=True)

PROMPT = "The key insight behind the transformer architecture is"

# Greedy — argmax at every step; deterministic, can repeat
print("greedy:", gen(PROMPT, max_new_tokens=40, do_sample=False))

# Temperature — divide logits by temp before softmax
#   temp < 1 → sharper (less random)   temp > 1 → flatter (more random)
print("temp=0.3:", gen(PROMPT, max_new_tokens=40, do_sample=True, temperature=0.3))
print("temp=1.5:", gen(PROMPT, max_new_tokens=40, do_sample=True, temperature=1.5))

# Top-k — zero out all but the k highest-prob tokens, then sample
print("top_k=20:", gen(PROMPT, max_new_tokens=40, do_sample=True, top_k=20, temperature=0.8))

# Top-p (nucleus) — keep the smallest set covering cumulative prob >= p
print("top_p=0.9:", gen(PROMPT, max_new_tokens=40, do_sample=True, top_p=0.9, temperature=0.8))

### Temperature reshapes the distribution

At temp → 0 the softmax collapses to a one-hot on the argmax (greedy). At temp → ∞ it flattens to uniform. Rule of thumb: `0.7` for creative tasks, `0.1` for factual/code.

In [ ]:
ids = tokenizer(PROMPT, return_tensors="pt").to(device)
with torch.no_grad():
    logits = model(**ids).logits[0, -1, :]   # last-position logits, [vocab_size]

for temp in [0.1, 0.5, 1.0, 2.0]:
    probs = F.softmax(logits / temp, dim=-1).topk(5)
    toks  = [tokenizer.decode([i]) for i in probs.indices.tolist()]
    print(f"temp={temp}  top-5 probs={probs.values.round(decimals=3).tolist()}  {toks}")

## Part B — Chat Templating

Instruct models are trained on a specific conversation format. Feeding raw text instead of a templated prompt puts the model out of distribution — responses degrade immediately. `add_generation_prompt=True` appends the model's assistant-turn prefix.

In [ ]:
messages = [
    {"role": "system",  "content": "You are a concise ML tutor. Answer in 2–3 sentences."},
    {"role": "user",    "content": "What problem does the residual connection solve?"},
]

prompt = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
print(repr(prompt[:180]))   # inspect the ChatML tokens

input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)
with torch.no_grad():
    out = model.generate(
        input_ids, max_new_tokens=80,
        do_sample=True, temperature=0.7, top_p=0.9,
        pad_token_id=tokenizer.eos_token_id,
    )
print(tokenizer.decode(out[0, input_ids.shape[1]:], skip_special_tokens=True))

## Part C — LoRA Fine-Tuning

Build a tiny instruction dataset, generate a baseline response, then run a full LoRA fine-tune loop with `peft` + `trl` and compare generation before vs. after. With only 8 examples the change is small — the workflow is what matters here.

In [ ]:
from datasets import Dataset
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

# ── tiny instruction dataset ──────────────────────────────────────────────
RAW = [
    ("What is gradient descent?",
     "Gradient descent updates parameters by moving them opposite to the gradient of the loss."),
    ("Explain the residual connection.",
     "A residual connection adds the block input to its output, giving gradients a clean bypass path."),
    ("What does RMSNorm do?",
     "RMSNorm normalizes by root-mean-square instead of mean+std, dropping the bias and being cheaper."),
    ("What is SwiGLU?",
     "SwiGLU multiplies a SiLU-activated linear projection by a second gate projection — empirically better per-parameter than a plain MLP."),
    ("What is the KV cache?",
     "The KV cache stores key/value tensors from prior steps so attention only computes them once per token."),
    ("Explain LoRA.",
     "LoRA freezes the base model and adds low-rank matrices A and B to target layers; only A and B are trained."),
    ("What is top-p sampling?",
     "Top-p sampling keeps the smallest token set whose cumulative probability reaches p, then samples from it."),
    ("What is RoPE?",
     "RoPE rotates Q and K by position-dependent angles so attention scores encode relative offsets."),
]

def _fmt(u, a):
    return tokenizer.apply_chat_template(
        [{"role": "user", "content": u}, {"role": "assistant", "content": a}],
        tokenize=False, add_generation_prompt=False,
    )

ds = Dataset.from_dict({"text": [_fmt(u, a) for u, a in RAW]})

# ── generate BEFORE ──────────────────────────────────────────────────────
TEST = "Explain LoRA."
before = gen(
    tokenizer.apply_chat_template([{"role":"user","content":TEST}],
                                  tokenize=False, add_generation_prompt=True),
    max_new_tokens=60, do_sample=False,
)
print("BEFORE:", before)

# ── LoRA config ──────────────────────────────────────────────────────────
lora_cfg = LoraConfig(
    r=8, lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05, bias="none",
    task_type="CAUSAL_LM",
)

sft_cfg = SFTConfig(
    output_dir="./qwen-lora-lab",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    bf16=(device == "cuda"),
    logging_steps=5,
    save_strategy="no",
    max_seq_length=256,
    dataset_text_field="text",
    report_to="none",
)

trainer = SFTTrainer(
    model=model, args=sft_cfg,
    train_dataset=ds, peft_config=lora_cfg,
)
trainer.train()

# ── generate AFTER ───────────────────────────────────────────────────────
model.eval()
after = gen(
    tokenizer.apply_chat_template([{"role":"user","content":TEST}],
                                  tokenize=False, add_generation_prompt=True),
    max_new_tokens=60, do_sample=False,
)
print("AFTER:", after)
# With 8 examples the change is small — the workflow is what matters here.
# Scale to 1k–10k examples for real adaptation.

## What you built

- Greedy, temperature, top-k, and top-p decoding with a direct look at how temperature reshapes the logit distribution.
- Correct chat templating — `apply_chat_template` for both inference and training data formatting.
- A full LoRA fine-tune loop: `LoraConfig`, `SFTConfig`, `SFTTrainer`, before/after generation on the same prompt.
- Device-aware dtype selection: bfloat16 on CUDA, float32 fallback on CPU/MPS.

## Stacks & alternatives

The lesson covers zero-Python serving with **Ollama**, high-throughput serving with **vLLM**, Apple-Silicon-native inference with **MLX / mlx-lm**, and faster/leaner LoRA with **Unsloth** (or config-driven **Axolotl**). See **`11_lab_llms.md`** for the code and the "Build it further" perplexity exercise.